In [1]:
import pandas as pd
import json

# Загрузка данных из JSON файлов
with open('ЦОН.json', 'r', encoding='utf-8') as file:
    c_on_data = json.load(file)

with open('hospitals_almaty.json', 'r', encoding='utf-8') as file:
    hospitals_data = json.load(file)

# Функция для извлечения данных и преобразования их в формат DataFrame
def extract_reviews(data, category, city='Алматы'):
    extracted_data = []
    for entry in data:
        # Основная информация об учреждении
        institution_info = {
            'id': entry.get('id'),
            'name': entry.get('name'),
            'category': category,  # ЦОН, больница и т.д.
            'city': city,
            'district': entry.get('address_name', '').split(',')[0],  # Пример разбивки по району, если есть
            'latitude': entry.get('point', {}).get('lat'),
            'longitude': entry.get('point', {}).get('lon'),
            'rating': entry.get('reviews', {}).get('general_rating'),
            'review_count': entry.get('reviews', {}).get('general_review_count'),
            'work_hours': entry.get('schedule', {}),
            'address': entry.get('address_name'),
            'tags': ', '.join([rubric.get('name') for rubric in entry.get('rubrics', [])])
        }
        
        # Обработка отзывов (каждый отзыв как отдельная строка)
        for review in entry.get('reviews_detailed', []):
            review_info = institution_info.copy()  # Копируем информацию об учреждении
            review_info.update({
                'office_id': entry.get('id'),
                'review_text': review.get('text'),
                'review_rating': review.get('rating'),
                'review_date': review.get('date', 'N/A')  # Если даты нет, можно использовать 'N/A'
            })
            extracted_data.append(review_info)
    
    return pd.DataFrame(extracted_data)

# Преобразуем данные для ЦОН и больниц
c_on_reviews_df = extract_reviews(c_on_data, category='ЦОН')
hospitals_reviews_df = extract_reviews(hospitals_data, category='Больница')

# Объединяем данные
combined_reviews_df = pd.concat([c_on_reviews_df, hospitals_reviews_df], ignore_index=True)

# Сохраняем результат в CSV
csv_file_path = 'almaty_centers_and_hospitals_reviews.csv'
combined_reviews_df.to_csv(csv_file_path, index=False)

# Печатаем путь к файлу для проверки
print(f"CSV file has been saved to {csv_file_path}")


CSV file has been saved to almaty_centers_and_hospitals_reviews.csv


In [2]:
df = pd.read_csv("almaty_centers_and_hospitals_reviews.csv")

In [3]:
df.head()

,id,name,category,city,district,latitude,longitude,rating,review_count,work_hours,address,tags,office_id,review_text,review_rating,review_date
0,9429940001088063,Специализированный центр обслуживания населени...,ЦОН,Алматы,улица Ак-Кайнар,43.308773,76.838086,3.6,2880,"{'Fri': {'working_hours': [{'from': '09:00', '...","улица Ак-Кайнар, 1",ЦОН,9429940001088063,Бардак. Номер қателесіп беріп жіберген. Өзің б...,1.0,NaN
1,9429940001088063,Специализированный центр обслуживания населени...,ЦОН,Алматы,улица Ак-Кайнар,43.308773,76.838086,3.6,2880,"{'Fri': {'working_hours': [{'from': '09:00', '...","улица Ак-Кайнар, 1",ЦОН,9429940001088063,Здраствуйте хочу рассказать о случае в котором...,1.0,NaN
2,9429940001088063,Специализированный центр обслуживания населени...,ЦОН,Алматы,улица Ак-Кайнар,43.308773,76.838086,3.6,2880,"{'Fri': {'working_hours': [{'from': '09:00', '...","улица Ак-Кайнар, 1",ЦОН,9429940001088063,"Нервные сотрудники, 5 часов ожиданий в пустую,...",1.0,NaN
3,9429940001088063,Специализированный центр обслуживания населени...,ЦОН,Алматы,улица Ак-Кайнар,43.308773,76.838086,3.6,2880,"{'Fri': {'working_hours': [{'from': '09:00', '...","улица Ак-Кайнар, 1",ЦОН,9429940001088063,Здрово!,4.0,NaN
4,9429940001088063,Специализированный центр обслуживания населени...,ЦОН,Алматы,улица Ак-Кайнар,43.308773,76.838086,3.6,2880,"{'Fri': {'working_hours': [{'from': '09:00', '...","улица Ак-Кайнар, 1",ЦОН,9429940001088063,Барлығы өз ісінің маманы,5.0,NaN


In [4]:
df.shape

(23661, 16)

In [5]:
df['name'].value_counts()

name
Специализированный центр обслуживания населения для автомобилистов г. Алматы, филиал №1    2856
Правительство для граждан Ауэзовского района г. Алматы, государственная корпорация         1744
Правительство для граждан Алмалинского района г. Алматы, государственная корпорация        1670
Городская больница скорой неотложной помощи                                                1614
Правительство для граждан Медеуского района г. Алматы, государственная корпорация          1448
Специализированный центр обслуживания населения для автомобилистов г. Алматы, филиал №4    1425
Центральная городская клиническая больница                                                 1398
Городская клиническая больница №5                                                          1340
Городская клиническая больница №7                                                          1302
Сункар, медицинский центр                                                                  1276
Правительство для граждан Бостандык

In [6]:
df.describe()

,id,latitude,longitude,rating,review_count,office_id,review_rating,review_date
count,2.366100e+04,23661.000000,23661.000000,23661.000000,23661.000000,2.366100e+04,23661.000000,0.0
mean,2.380384e+16,43.256310,76.890699,3.779447,1441.309328,2.380384e+16,3.035544,NaN
std,2.576911e+16,0.044245,0.056224,0.554513,636.800394,2.576911e+16,1.902162,NaN
min,9.429940e+15,43.206951,76.790634,2.600000,475.000000,9.429940e+15,1.000000,NaN
25%,9.429940e+15,43.228526,76.838086,3.500000,916.000000,9.429940e+15,1.000000,NaN
50%,9.429940e+15,43.233299,76.900985,3.800000,1403.000000,9.429940e+15,3.000000,NaN
75%,9.429940e+15,43.307362,76.932978,4.100000,1691.000000,9.429940e+15,5.000000,NaN
max,7.000000e+16,43.365744,77.013877,4.700000,2880.000000,7.000000e+16,5.000000,NaN


In [7]:
df.isna().sum() 

id                   0
name                 0
category             0
city                 0
district             0
latitude             0
longitude            0
rating               0
review_count         0
work_hours           0
address              0
tags                 0
office_id            0
review_text          0
review_rating        0
review_date      23661
dtype: int64

In [8]:
import pandas as pd
import numpy as np

# Загрузка данных
df = pd.read_csv('almaty_centers_and_hospitals_reviews.csv')

# Преобразование данных

# 2. Приведем категорию к нижнему регистру
df['category'] = df['category'].str.lower()

# 3. Очистим данные по городу (если есть вариации)
df['city'] = df['city'].str.strip().apply(lambda x: 'Алматы' if x.lower() != 'алматы' else x)

# 4. Очистим колонку с районом/улицей, оставим только название
df['district'] = df['district'].str.strip().apply(lambda x: x.split(',')[0])

# 5. Преобразуем координаты в числовой формат, заменяя некорректные значения на NaN
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

# 6. Преобразуем рейтинг в числовой формат
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

# 7. Преобразуем количество отзывов в числовой формат
df['review_count'] = pd.to_numeric(df['review_count'], errors='coerce')

# 8. Часы работы приведем к строковому формату "09:00-18:00" или сумме часов в неделю

# Функция для извлечения рабочих часов
def extract_working_hours(work_hours):
    # Проверим, является ли work_hours строкой JSON
    if isinstance(work_hours, str):
        try:
            # Попробуем преобразовать строку JSON в словарь
            work_hours = json.loads(work_hours)
        except json.JSONDecodeError:
            # Если ошибка декодирования, возвращаем N/A
            return 'N/A'
    
    hours = []
    if isinstance(work_hours, dict):
        # Проходим по рабочим дням и извлекаем рабочие часы
        for day, schedule in work_hours.items():
            for period in schedule.get('working_hours', []):
                hours.append(f"{period['from']}-{period['to']}")
    return ', '.join(hours) if hours else 'N/A'

# Применим функцию к столбцу 'work_hours'
df['work_hours'] = df['work_hours'].apply(extract_working_hours)


# 10. Преобразуем теги в строку, разделенную запятыми
df['tags'] = df['tags'].str.join(', ') if isinstance(df['tags'], list) else df['tags']


# 12. Преобразуем рейтинг отзыва в числовой формат
df['review_rating'] = pd.to_numeric(df['review_rating'], errors='coerce')

# Проверим результат после очистки
df.head()


,id,name,category,city,district,latitude,longitude,rating,review_count,work_hours,address,tags,office_id,review_text,review_rating,review_date
0,9429940001088063,Специализированный центр обслуживания населени...,цон,Алматы,улица Ак-Кайнар,43.308773,76.838086,3.6,2880,N/A,"улица Ак-Кайнар, 1",ЦОН,9429940001088063,Бардак. Номер қателесіп беріп жіберген. Өзің б...,1.0,NaN
1,9429940001088063,Специализированный центр обслуживания населени...,цон,Алматы,улица Ак-Кайнар,43.308773,76.838086,3.6,2880,N/A,"улица Ак-Кайнар, 1",ЦОН,9429940001088063,Здраствуйте хочу рассказать о случае в котором...,1.0,NaN
2,9429940001088063,Специализированный центр обслуживания населени...,цон,Алматы,улица Ак-Кайнар,43.308773,76.838086,3.6,2880,N/A,"улица Ак-Кайнар, 1",ЦОН,9429940001088063,"Нервные сотрудники, 5 часов ожиданий в пустую,...",1.0,NaN
3,9429940001088063,Специализированный центр обслуживания населени...,цон,Алматы,улица Ак-Кайнар,43.308773,76.838086,3.6,2880,N/A,"улица Ак-Кайнар, 1",ЦОН,9429940001088063,Здрово!,4.0,NaN
4,9429940001088063,Специализированный центр обслуживания населени...,цон,Алматы,улица Ак-Кайнар,43.308773,76.838086,3.6,2880,N/A,"улица Ак-Кайнар, 1",ЦОН,9429940001088063,Барлығы өз ісінің маманы,5.0,NaN
